In [10]:
import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

### Data Import and Quick Clean

In [11]:
data = pd.read_csv("ELA_data_2013_2023.csv")
school_poverty = pd.read_excel('Demographic_Snapshot_2017-18_to_2021-22__Public_.xlsx', sheet_name = 'School')

In [12]:
from data_cleaning import cleaning_data, change_variable_type, create_school_level_data, merge_data_and_poverty, categorize_poverty, categorize_title_i, categorize_economic_need
# clean data
school_data = cleaning_data(data) 

In [13]:
# convert num columns
cols = ['mean_scale_score','level_1_count','level_2_count','level_3_count', 'level_4_count', 'level_4_percentage', 'level_3_4_count',
        'level_1_percentage','level_2_percentage','level_3_percentage','level_4_percentage','level_3_4_percentage']
school_data = change_variable_type(school_data,cols)
# change year to object
school_data['Year'] = school_data['Year'].astype('object')


# create school level data, all grades all students
school_level_data = create_school_level_data(school_data)

# merge with poverty data
merged_data = merge_data_and_poverty(school_level_data, school_poverty)

# convert poverty percentage to float and categorize, round to 3 decimals
merged_data['% Poverty'] = merged_data['% Poverty'].astype(float)
merged_data['Economic Need Index'] = merged_data['Economic Need Index'].astype(float)

merged_data['Poverty_Category'] = merged_data.apply(categorize_poverty, axis=1)
merged_data['Economic_Need_Index_Category'] = merged_data.apply(categorize_economic_need, axis=1)

merged_data['% Poverty'] = merged_data['% Poverty'].round(3)
merged_data['Economic Need Index'] = merged_data['Economic Need Index'].round(3)

In [14]:
merged_data

,Report Category,DBN,school_name,Grade,Year,Student Category,number_tested,mean_scale_score,level_1_count,level_1_percentage,...,level_3_count,level_3_percentage,level_4_count,level_4_percentage,level_3_4_count,level_3_4_percentage,% Poverty,Economic Need Index,Poverty_Category,Economic_Need_Index_Category
0,School,01M015,P.S. 015 ROBERTO CLEMENTE,All Grades,2022,All Students,74,596.0,21.0,28.4,...,13.0,17.6,10.0,13.5,23.0,31.1,0.845,0.888,1,1
1,School,01M015,P.S. 015 ROBERTO CLEMENTE,All Grades,2019,All Students,70,606.0,7.0,10.0,...,33.0,47.1,9.0,12.9,42.0,60.0,0.845,0.888,1,1
2,School,01M020,P.S. 020 ANNA SILVER,All Grades,2022,All Students,110,589.0,48.0,43.6,...,15.0,13.6,11.0,10.0,26.0,23.6,0.678,0.754,0,1
3,School,01M020,P.S. 020 ANNA SILVER,All Grades,2019,All Students,197,597.0,50.0,25.4,...,58.0,29.4,20.0,10.2,78.0,39.6,0.678,0.754,0,1
4,School,01M034,P.S. 034 FRANKLIN D. ROOSEVELT,All Grades,2022,All Students,151,589.0,60.0,39.7,...,23.0,15.2,10.0,6.6,33.0,21.9,0.950,0.948,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2225,School,32K384,P.S. /I.S. 384 FRANCES E. CARTER,All Grades,2019,All Students,290,589.0,117.0,40.3,...,66.0,22.8,12.0,4.1,78.0,26.9,0.867,0.903,1,1
2226,School,32K554,ALL CITY LEADERSHIP SECONDARY SCHOOL,All Grades,2022,All Students,157,620.0,7.0,4.5,...,34.0,21.7,105.0,66.9,139.0,88.5,0.816,0.693,1,0
2227,School,32K554,ALL CITY LEADERSHIP SECONDARY SCHOOL,All Grades,2019,All Students,173,621.0,4.0,2.3,...,45.0,26.0,113.0,65.3,158.0,91.3,0.816,0.693,1,0
2228,School,32K562,EVERGREEN MIDDLE SCHOOL FOR URBAN EXPLORATION,All Grades,2022,All Students,245,596.0,76.0,31.0,...,63.0,25.7,35.0,14.3,98.0,40.0,0.949,0.906,1,1


### Model, testing differences with Grade 3

In [15]:
school_data_3 = school_data[
    (school_data['Grade'] == '3') & 
    (school_data['Student Category'] == 'All Students') &
    (school_data['Report Category'] == 'School')
]

#### Using Poverty compared against NYC mean

In [16]:
merged_data_poverty = merge_data_and_poverty(school_data_3, school_poverty)

# fix poverty percentage to float, categorize, round to 3 decimals
merged_data_poverty['% Poverty'] = merged_data_poverty['% Poverty'].astype(float)
merged_data_poverty['Poverty_Category'] = merged_data_poverty.apply(categorize_poverty, axis=1)
merged_data_poverty['% Poverty'] = merged_data_poverty['% Poverty'].round(3)

# set index for panel data, filter for schools with 2 years of data (2019 and 2022)
data_poverty= merged_data_poverty.set_index(['school_name','Year'])

counts = data_poverty.groupby(level=0).size() # check count of years for each school
valid_schools = counts[counts == 2].index # filter data to include only valid schools
data_poverty_adjust = data_poverty.loc[valid_schools]

data_poverty_adjust['Poverty_Category'] = data_poverty_adjust['Poverty_Category'].astype(int)

# vars for did model
data_poverty_adjust['treated'] = (data_poverty_adjust['Poverty_Category'] == 1).astype(int)
data_poverty_adjust['post'] = (data_poverty_adjust.index.get_level_values('Year') == 2022).astype(int)
data_poverty_adjust['did'] = (data_poverty_adjust['treated'] * data_poverty_adjust['post']).astype(int)

In [17]:
mod = PanelOLS.from_formula('mean_scale_score ~ 1 + did + EntityEffects + TimeEffects', data_poverty_adjust, weights=data_poverty_adjust['number_tested'])
res = mod.fit(cov_type='clustered', cluster_entity=True)

# can include weights, excluded for now for baseline model
# weights=data_poverty_adjust['number_tested']

print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0209
Estimator:                   PanelOLS   R-squared (Between):              0.0424
No. Observations:                1536   R-squared (Within):               0.0658
Date:                Thu, Mar 05 2026   R-squared (Overall):              0.0441
Time:                        23:00:02   Log-likelihood                   -3617.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   3.381e+06
Entities:                         771   P-value                           0.0000
Avg Obs:                       1.9922   Distribution:                   F(1,763)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             7.6630
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


#### Using Title I Eligibility 

In [18]:
merged_data_titlei = merge_data_and_poverty(school_data_3, school_poverty)

# fix poverty percentage to float, categorize, round to 3 decimals
merged_data_titlei['% Poverty'] = merged_data_titlei['% Poverty'].astype(float)
merged_data_titlei['Title I Category'] = merged_data_titlei.apply(categorize_title_i, axis=1)
# merged_data_titlei['% Poverty'] = merged_data_titlei['% Poverty'].round(3)

# set index for panel data, filter for schools with 2 years of data (2019 and 2022)
data_titlei= merged_data_titlei.set_index(['school_name','Year'])

counts = data_titlei.groupby(level=0).size() # check count of years for each school
valid_schools = counts[counts == 2].index # filter data to include only valid schools
data_titlei_adjust = data_titlei.loc[valid_schools]

data_titlei_adjust['Title I Category'] = data_titlei_adjust['Title I Category'].astype(int)

# vars for did model
data_titlei_adjust['treated'] = (data_titlei_adjust['Title I Category'] == 1).astype(int)
data_titlei_adjust['post'] = (data_titlei_adjust.index.get_level_values('Year') == 2022).astype(int)
data_titlei_adjust['did'] = (data_titlei_adjust['treated'] * data_titlei_adjust['post']).astype(int)

In [21]:
mod = PanelOLS.from_formula('mean_scale_score ~ 1 + did + EntityEffects + TimeEffects', data_titlei_adjust, weights=data_titlei_adjust['number_tested'])
res = mod.fit(cov_type='clustered', cluster_entity=True)


print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0309
Estimator:                   PanelOLS   R-squared (Between):              0.0477
No. Observations:                1536   R-squared (Within):               0.0694
Date:                Thu, Mar 05 2026   R-squared (Overall):              0.0493
Time:                        23:00:24   Log-likelihood                   -3609.6
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   3.416e+06
Entities:                         771   P-value                           0.0000
Avg Obs:                       1.9922   Distribution:                   F(1,763)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             13.228
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


#### Using Economic Need Index compared against NYC mean

In [ ]:
merged_data_eni = merge_data_and_poverty(school_data_3, school_poverty)

# fix eni percentage to float, categorize, round to 3 decimals
merged_data_eni['Economic Need Index'] = merged_data_eni['Economic Need Index'].astype(float)
merged_data_eni['Economic Need Category'] = merged_data_eni.apply(categorize_economic_need, axis=1)
# merged_data_eni['% Poverty'] = merged_data_eni['% Poverty'].round(3)

# set index for panel data, filter for schools with 2 years of data (2019 and 2022)
data_eni= merged_data_eni.set_index(['school_name','Year'])

counts = data_eni.groupby(level=0).size() # check count of years for each school
valid_schools = counts[counts == 2].index # filter data to include only valid schools
data_eni_adjust = data_eni.loc[valid_schools]

data_eni_adjust['Economic Need Category'] = data_eni_adjust['Economic Need Category'].astype(int)

# vars for did model
data_eni_adjust['treated'] = (data_eni_adjust['Economic Need Category'] == 1).astype(int)
data_eni_adjust['post'] = (data_eni_adjust.index.get_level_values('Year') == 2022).astype(int)
data_eni_adjust['did'] = (data_eni_adjust['treated'] * data_eni_adjust['post']).astype(int)

In [23]:
mod = PanelOLS.from_formula('mean_scale_score ~ 1 + did + EntityEffects + TimeEffects', data_eni_adjust, weights=data_eni_adjust['number_tested'])
res = mod.fit(cov_type='clustered', cluster_entity=True)


print(res.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:       mean_scale_score   R-squared:                        0.0200
Estimator:                   PanelOLS   R-squared (Between):              0.0419
No. Observations:                1536   R-squared (Within):               0.0638
Date:                Fri, Mar 06 2026   R-squared (Overall):              0.0435
Time:                        11:37:40   Log-likelihood                   -3618.2
Cov. Estimator:             Clustered                                           
                                        F-statistic:                   3.378e+06
Entities:                         771   P-value                           0.0000
Avg Obs:                       1.9922   Distribution:                   F(1,763)
Min Obs:                       1.0000                                           
Max Obs:                       2.0000   F-statistic (robust):             7.3773
                            

c:\Users\madis\AppData\Local\Programs\Python\Python313\Lib\site-packages\linearmodels\panel\model.py:1258: MissingValueWarning: 
Inputs contain missing values. Dropping rows with missing observations.
  super().__init__(dependent, exog, weights=weights, check_rank=check_rank)


### Comparing Treatment Variable Methods

When trying to classify poverty, there are three methods that were tested; comparing school-based poverty percentage against the NYC mean, comparing school-based poverty percentage against the NYC Department of Education's (DoE) Title I eligibility, and comparing the NYC DoE Economic Need Index (ENI) against the NYC mean. <br>

Comparing school-based poverty percentage was the initial plan for this project. This provides a simple method to classify schools as impoverished or not but may not fully encompass external factors that Title I eligibility and the ENI do. While results using this method were significant and aligned with the overall hypothesis, the generality of the categorization raised red flags towards the result. <br>

Upon further research, I found that the NYC DoE used a different metric to qualify students for Title I eligibility. For those who are unaware, Title I is a federal program that provides funding to schools that have a high concentration of low-income students. Compared to the intial method, the eligibility percentage is below the overall NYC poverty rate, with schools who have a poverty rate of 60% are classified as in need of funding. This method was the best performing overall, with the lowest p-value and highest R-squared. <br>

The final method that was investigated was the usage of the NYC DoE's Economic Need Index. Luckily, my demographic data already included this information. According to the NYC DoE Performance Dashboard, the "Economic Need Index is an estimate of the percentage of students at the school facing economic hardship, based on temporary housing, eligibility for public assistance, and Census tract poverty rates." This method is very similar to comparing the poverty percentage of a school to the mean. However, the ENI incorporates more external factors into this estimate. This method yielded the second-best results. <br>

The method used depends on how saturated I want the poverty binary variable to be. Comparing school-wide poverty to the average of the city is a simplified method of categorizing poverty overall. However, this method fails to consider schools that are on the cusp of the poverty average and may not account for schools that may be struggling according to the DoE. <br>

Going forward, the Title I eligibility will be the method used to categorize the schools. For the sake of the baseline models, only Grade 3 model performance was assessed due to the hypothesis of this being the most affected group. As modelling progresses for all grades, this method may be subject to change, in which case I plan on further exploring the usage of the Economic Need Index.




#### Sources

https://data.cccnewyork.org/data/bar/1371/student-economic-need-index#1371/a/1/1622/127